In [2]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
# import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

In [3]:
treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    matrix_type="coverage",
)

In [ ]:
X = corpus.feature_matrix
print("Feature matrix shape:", X.shape)

In [ ]:
X

In [ ]:
import skfuzzy as fuzz
import numpy as np
Xd = np.asarray(X, dtype=float)
n_clusters = 10 # obtained with dunn and db index
m = 2 
# scikit-fuzzy expects (features, samples), so transpose
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    Xd.T, c=n_clusters, m=m, error=1e-5, maxiter=1000, init=None, seed=42
)
labels = u.argmax(axis=0)

In [ ]:
membership = u.T
membership

In [ ]:
fpc

In [ ]:
dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)

In [ ]:
fig = tod.plotting.fuzzy_cluster_scatter_plot(corpus, dim_red, membership)

In [ ]:
fig.write_html("fuzzy_cluster_scatter_plot.html")

In [ ]:
def fuzzy_multi_cluster_lexunits(corpus: Corpus, membership: np.ndarray, threshold: float = 0.3):
    """
    Return list of (lexunit, memberships_dict) for lexical units having membership
    >= threshold in at least 2 clusters.
    membership shape: (n_samples, n_clusters)
    """
    result = []
    for i, row in enumerate(membership):
        passed = {c: float(m) for c, m in enumerate(row) if m >= threshold}
        if len(passed) >= 2:
            result.append((corpus.idx2lexunit(i), passed))
    return result

def fuzzy_pair_overlaps(corpus: Corpus, membership: np.ndarray, threshold: float = 0.3):
    """
    Return dict mapping (cluster_a, cluster_b) -> list of lexunits that have membership
    >= threshold in both clusters.
    """
    n_clusters = membership.shape[1]
    overlaps = {}
    for i, row in enumerate(membership):
        active = [c for c, m in enumerate(row) if m >= threshold]
        if len(active) >= 2:
            lex = corpus.idx2lexunit(i)
            for a in range(len(active)):
                for b in range(a + 1, len(active)):
                    pair = (active[a], active[b])
                    overlaps.setdefault(pair, []).append(lex)
    return overlaps

def fuzzy_ambiguous_lexunits(corpus: Corpus, membership: np.ndarray, top_k: int = 2, max_gap: float = 0.15):
    """
    Return lexical units whose top_k memberships are close (difference between
    rank 1 and rank top_k <= max_gap).
    """
    result = []
    for i, row in enumerate(membership):
        order = np.argsort(row)[::-1]
        top_vals = row[order][:top_k]
        if len(top_vals) == top_k and (top_vals[0] - top_vals[-1]) <= max_gap:
            result.append((corpus.idx2lexunit(i), [(int(c), float(row[c])) for c in order[:top_k]]))
    return result

In [ ]:
# Get units with membership >= 0.35 in at least two clusters
multi = fuzzy_multi_cluster_lexunits(corpus, membership, threshold=0.35)
print(f"{len(multi)} multi-cluster lexical units")
print(multi[:10])

# Pairs overlap
pairs = fuzzy_pair_overlaps(corpus, membership, threshold=0.35)
for pair, lexunits in list(pairs.items())[:5]:
    print("Pair", pair, "count", len(lexunits))

# Ambiguous (close top 2 memberships)
amb = fuzzy_ambiguous_lexunits(corpus, membership, top_k=2, max_gap=0.1)
print(f"{len(amb)} ambiguous lexical units")
print(amb[:10])

In [ ]:
print("Membership per row variance (first 10):",
      [float(np.var(r)) for r in membership[:10]])
print("All membership rows identical? ",
      np.allclose(membership, membership[0]))
print("Row sums (should be 1):", membership.sum(axis=1)[:5])
print("Cluster center pairwise max diff:",
      np.max([np.max(np.abs(c1 - c2)) for i,c1 in enumerate(cntr) for j,c2 in enumerate(cntr) if i<j]))
print("Per-feature std (first 10):", np.std(Xd, axis=0)[:10])
print("Zero-variance feature count:", np.sum(np.std(Xd, axis=0)==0))

In [ ]:
best = None
for c in range(2, 14):
    _, u, _, _, _, _, fpc = fuzz.cluster.cmeans(Xd.T, c=c, m=2.0, error=1e-5, maxiter=1000, seed=42)
    if best is None or fpc > best[0]:
        best = (fpc, c, u)
print(f"Best c by FPC: {best[1]} (FPC={best[0]:.4f})")

In [ ]:
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

# Dense and safe
Xd = X.toarray() if hasattr(X, "toarray") else np.asarray(X, dtype=float)
Xd = np.nan_to_num(Xd, copy=False)

# 1) Drop lowest-variance 20% of features (keeps structure, removes near-constant cols)
col_std = Xd.std(axis=0)
keep = col_std > np.percentile(col_std, 20)
Xd1 = Xd[:, keep]

# 2) Row L2-normalize (approximates cosine distance)
Xd1 = normalize(Xd1, axis=1)

# 3) Scale columns (avoid mean-centering to keep sparsity semantics)
Xd2 = StandardScaler(with_mean=False).fit_transform(Xd1)

# 4) Dimensionality reduction (denoise + avoid collinearity)
n_comp = min(100, Xd2.shape[1]-1) if Xd2.shape[1] > 50 else max(2, Xd2.shape[1])
if n_comp < Xd2.shape[1]:
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    Xf = svd.fit_transform(Xd2)
else:
    Xf = Xd2

print("Preprocessed shape:", Xf.shape)

# 5) Choose clusters
n_clusters = 8

# 6) KMeans to build an initial membership matrix (one-hot)
km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(Xf)
labels0 = km.labels_                            # length = n_samples
u0_init = np.zeros((n_clusters, Xf.shape[0]))   # (c, n_samples)
u0_init[labels0, np.arange(Xf.shape[0])] = 1.0

# 7) Fuzzy c-means (expects features x samples)
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    Xf.T, c=n_clusters, m=2.0, error=1e-5, maxiter=1000, init=u0_init, seed=42
)

membership = u.T             # shape: (n_words, n_clusters)
labels = membership.argmax(axis=1)

print("FPC:", fpc, " avg membership row var:", float(membership.var(axis=1).mean()))
print("Cluster sizes:", np.bincount(labels))

In [4]:
clustering = tod.clustering.SparseKMeans(corpus=corpus, k=10, top_n_features=10, cluster_defining_features=True)
centroids = clustering.get_centroids(corpus)

Cluster 0 - Top 10 defining features:
  1. node:X:own:rel_shallow=amod                        importance: 1.240142
  2. node:X:prev:upos=DET                               importance: 0.108083
  3. node:X:own:Gender=Fem                              importance: 0.100176
  4. node:X:own:Gender=Masc                             importance: 0.092085
  5. node:X:parent:upos=NOUN                            importance: 0.087285
  6. node:X:prev:PronType=Art                           importance: 0.079271
  7. node:X:parent:position=after                       importance: 0.050477
  8. node:X:own:Number=Plur                             importance: 0.044960
  9. node:X:own:Number=Sing                             importance: 0.036883
 10. node:X:prev:Definite=Def                           importance: 0.033560

Cluster 1 - Top 10 defining features:
  1. node:X:parent:upos=NOUN                            importance: 0.090193
  2. node:X:parent:position=after                       importance: 0.076444

In [38]:
cluster_def_features = clustering.cluster_features
cluster_def_features

{0: [(1182, 'node:X:own:rel_shallow=amod', 1.240142176759132),
  (1965, 'node:X:prev:upos=DET', 0.10808348087319848),
  (1134, 'node:X:own:Gender=Fem', 0.10017562693840869),
  (1135, 'node:X:own:Gender=Masc', 0.09208541493459108),
  (1593, 'node:X:parent:upos=NOUN', 0.08728497305312663),
  (1937, 'node:X:prev:PronType=Art', 0.07927132697602293),
  (1584, 'node:X:parent:position=after', 0.0504765070266343),
  (1141, 'node:X:own:Number=Plur', 0.04496014309822504),
  (1142, 'node:X:own:Number=Sing', 0.03688251524333891),
  (1904, 'node:X:prev:Definite=Def', 0.03355983325138406)],
 1: [(1593, 'node:X:parent:upos=NOUN', 0.0901933723020979),
  (1584, 'node:X:parent:position=after', 0.0764439301556618),
  (1967, 'node:X:prev:upos=NOUN', 0.06175190886234081),
  (1550, 'node:X:parent:Number=Sing', 0.028204933479713434),
  (798, 'node:X:next:upos=NOUN', 0.02251735928238496),
  (1543, 'node:X:parent:Gender=Masc', 0.020637219607487482),
  (1961, 'node:X:prev:upos=ADP', 0.020400287717617923),
  (15

In [10]:
feature_names = corpus._feature2idx.keys()

In [12]:
for k in cluster_def_features:
    defining_feats = cluster_def_features[k]
    print(defining_feats)
    defining = [feature_names[f] for f in defining_feats]
    print(defining)
    print("---")

[(1182, 'node:X:own:rel_shallow=amod', 1.240142176759132), (1965, 'node:X:prev:upos=DET', 0.10808348087319848), (1134, 'node:X:own:Gender=Fem', 0.10017562693840869), (1135, 'node:X:own:Gender=Masc', 0.09208541493459108), (1593, 'node:X:parent:upos=NOUN', 0.08728497305312663), (1937, 'node:X:prev:PronType=Art', 0.07927132697602293), (1584, 'node:X:parent:position=after', 0.0504765070266343), (1141, 'node:X:own:Number=Plur', 0.04496014309822504), (1142, 'node:X:own:Number=Sing', 0.03688251524333891), (1904, 'node:X:prev:Definite=Def', 0.03355983325138406)]


TypeError: 'dict_keys' object is not subscriptable

In [36]:
membership

array([[0.05028801, 0.05029294, 0.12820682, ..., 0.13425617, 0.11890324,
        0.04365771],
       [0.05350103, 0.0583354 , 0.09936953, ..., 0.14714015, 0.13307939,
        0.04263993],
       [0.11427567, 0.19492187, 0.06879178, ..., 0.0863174 , 0.0817302 ,
        0.06713025],
       ...,
       [0.00402016, 0.00204715, 0.00763395, ..., 0.05055663, 0.91129041,
        0.00182631],
       [0.02481153, 0.01794664, 0.0338334 , ..., 0.07227752, 0.07140636,
        0.01913966],
       [0.05059037, 0.05514522, 0.14531254, ..., 0.1280074 , 0.11331591,
        0.04204266]])

In [34]:
def explain_sample(idx, clusters_above_threshold, clustering, top_n=10):
    cluster_def_features = clustering.cluster_features
    print(clusters_above_threshold)
    x = Xd[idx]
    explanations = []
    for k in clusters_above_threshold:
        centroid = centroids[k]
        diff = x - centroid
        sq = diff ** 2
        total = sq.sum()
        contrib = sq / total if total > 0 else np.zeros_like(sq)

        # Order features by contribution (ascending)
        order = np.argsort(contrib)
        supportive_idx = order[:top_n]          # closest features
        opposing_idx = order[-top_n:][::-1]     # farthest features

        supportive = [(corpus.idx2feature(i), float(contrib[i])) for i in supportive_idx]
        opposing = [(corpus.idx2feature(i), float(contrib[i])) for i in opposing_idx]

        explanations.append({
            "cluster": int(k),
            "supportive_features": supportive,
            "opposing_features": opposing,
            })

        # def_features = [cluster_def_features[k][i][1] for i in len(cluster_def_features[k])]

        # print(def_features)
        # overlap = [name for name, _ in supportive if name in defining_names]

        # explanations.append({
        #     "cluster": int(k),
        #     "membership": float(membership[idx, k]),
        #     "supportive_features": supportive,
        #     "opposing_features": opposing,
        #     "defining_features_cluster": defining_names,
        #     "supportive_and_defining_overlap": overlap
        # })
    return explanations

In [ ]:
for i in range(len(df_multi)):
    sample_row = int(df_multi.iloc[i]["row_idx"])
    lex_unit = df_multi.iloc[i]["lexunit"]
    clusters_above_threshold = [c for c, _m in df_multi.iloc[i]["clusters_above_threshold"]]
    detailed = explain_sample(sample_row, clusters_above_threshold, clustering)

    import json
    with open("fuzzy_explanations_sample.json", "w") as fh:
        json.dump({
            "lexunit": lex_unit,
            "clusters_above_threshold": clusters_above_threshold,
            "explanation": detailed
        }, fh, indent=2)

    print(f"Saved per-feature explanation for sample row {sample_row} to fuzzy_explanations_sample.json")

[2, 4, 5, 6, 7, 8]
Saved per-feature explanation for sample row 0 to fuzzy_explanations_sample.json
[5, 6, 7, 8]
Saved per-feature explanation for sample row 1 to fuzzy_explanations_sample.json
[0, 1, 3, 4]
Saved per-feature explanation for sample row 2 to fuzzy_explanations_sample.json
[0, 1]
Saved per-feature explanation for sample row 3 to fuzzy_explanations_sample.json
[0, 1, 3]
Saved per-feature explanation for sample row 4 to fuzzy_explanations_sample.json
[1, 2, 6]
Saved per-feature explanation for sample row 5 to fuzzy_explanations_sample.json
[0, 7, 8]
Saved per-feature explanation for sample row 6 to fuzzy_explanations_sample.json
[0, 1, 7, 8]
Saved per-feature explanation for sample row 7 to fuzzy_explanations_sample.json
[7, 8]
Saved per-feature explanation for sample row 9 to fuzzy_explanations_sample.json
[0, 1, 7, 8]
Saved per-feature explanation for sample row 10 to fuzzy_explanations_sample.json
[7, 8]
Saved per-feature explanation for sample row 11 to fuzzy_explanatio

In [6]:
corpus._feature2idx

{'node:X:child:Cxn=Conditional-NegativeEpistemic': 0,
 'node:X:child:Cxn=Conditional-NeutralEpistemic': 1,
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Conditional-Reduced': 2,
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Existential-HavePred-ItExpl-ThereExpl': 3,
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Interrogative-Polar-Direct': 4,
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Interrogative-WHInfo-Direct': 5,
 'node:X:child:Cxn=Conditional-NeutralEpistemic,Interrogative-WHInfo-Indirect': 6,
 'node:X:child:Cxn=Existential-HavePred-ItExpl-ThereExpl': 7,
 'node:X:child:Cxn=Interrogative-Alternative': 8,
 'node:X:child:Cxn=Interrogative-Polar-Direct': 9,
 'node:X:child:Cxn=Interrogative-Polar-Indirect': 10,
 'node:X:child:Cxn=Interrogative-WHInfo-Direct': 11,
 'node:X:child:Cxn=Interrogative-WHInfo-Indirect': 12,
 'node:X:child:Cxn=NPN': 13,
 'node:X:child:CxnElt=10:Conditional-NeutralEpistemic.Apodosis@p': 14,
 'node:X:child:CxnElt=10:Conditional-NeutralEpistemic.Protas

In [4]:
centroids.shape

(10, 1976)

In [15]:
import numpy as np
import skfuzzy as fuzz

# Data as dense float, shape (n_samples, n_features)
X = corpus.feature_matrix
Xd = X.toarray() if hasattr(X, "toarray") else np.asarray(X, dtype=float)

# Centroids from SparseKMeans: shape (c, n_features)
centroids = np.asarray(centroids, dtype=float)

m = 1.5
# Use cmeans_predict with fixed centers
u, u0, d, jm, p, fpc = fuzz.cluster.cmeans_predict(
    Xd.T, centroids, m=m, error=1e-5, maxiter=1000, seed=42
)
labels = u.argmax(axis=0)


In [ ]:
import numpy as np
k_eff = np.exp((-membership * np.log(membership + 1e-12)).sum(axis=1))
print("Mean effective clusters:", float(k_eff.mean()))

In [8]:
fpc

0.6213022022089072

In [ ]:
import numpy as np
c = u.shape[0]
print("1/c baseline:", 1.0/c)
print("avg max membership:", float(u.T.max(axis=1).mean()))


In [ ]:
# Average membership entropy (lower => crisper)
import numpy as np
eps = 1e-12
entropy = (-membership * np.log(membership + eps)).sum(axis=1)
print("Mean entropy:", float(entropy.mean()), "max possible:", np.log(membership.shape[1]))

In [ ]:
membership = u.T
membership

In [50]:
import numpy as np
import pandas as pd

# Ensure membership is (n_samples, n_clusters)
membership = u.T  # if not already set

threshold = 0.1
hard_labels = membership.argmax(axis=1)  # hard (argmax) cluster per sample

mask = membership >= threshold
multi_rows = np.where(mask.sum(axis=1) >= 2)[0]  # row indices in original X

rows = []
for i in multi_rows:
    clusters = np.where(mask[i])[0]
    rows.append({
        "row_idx": int(i),                                   # row in X
        "lexunit": corpus.idx2lexunit(i),                    # optional: your item id
        "primary_cluster": int(hard_labels[i]),              # argmax cluster
        "clusters_above_threshold": [(int(c), float(membership[i, c])) for c in clusters]
    })

df_multi = pd.DataFrame(rows).sort_values("row_idx")
def cluster_fuzziness(probs):
    probs = np.asarray(probs)
    probs = probs[probs > 0]
    K = len(probs)
    # Normalized entropy
    H = -np.sum(probs * np.log(probs)) / np.log(K)
    # Max membership
    u_max = np.max(probs)
    # Ratio of top two
    sorted_probs = np.sort(probs)[::-1]
    ratio = sorted_probs[0] / sorted_probs[1] if len(sorted_probs) > 1 else np.inf
    return {
        "entropy": H,
        "one_minus_max": 1 - u_max,
        "top2_ratio": ratio
    }

df_multi[["entropy", "one_minus_max", "top2_ratio"]] = [
    pd.Series(cluster_fuzziness([p for _, p in row["clusters_above_threshold"]]))
    for _, row in df_multi.iterrows()
]

print(f"{len(df_multi)} items are in >=2 clusters (threshold={threshold}).")
display(df_multi)

927 items are in >=2 clusters (threshold=0.1).


,row_idx,lexunit,primary_cluster,clusters_above_threshold,entropy,one_minus_max,top2_ratio
0,0,"($, NOUN)",6,"[(2, 0.128206817996798), (4, 0.100318394796285...",0.888183,0.839418,1.100048
1,1,"(%, NOUN)",5,"[(5, 0.15485693440929038), (6, 0.1296266755976...",0.796409,0.845143,1.052445
2,2,"(&, CCONJ)",1,"[(0, 0.11427567214695541), (1, 0.1949218652084...",0.765647,0.805078,1.635136
3,3,"(/, ADP)",1,"[(0, 0.14366121927568976), (1, 0.4049688184573...",0.930272,0.595031,2.818915
4,4,"(/, CCONJ)",1,"[(0, 0.11638331436529192), (1, 0.3117716558035...",0.800305,0.688228,2.392811
...,...,...,...,...,...,...,...
922,2927,"(être, AUX)",3,"[(0, 0.11009323952659923), (1, 0.1328488216265...",0.821283,0.777806,1.388416
923,2929,"(île, NOUN)",8,"[(7, 0.11921702901103254), (8, 0.8257710922698...",0.593865,0.174229,6.926620
924,2930,"(œil, NOUN)",7,"[(7, 0.6934441378352377), (8, 0.15738746145034...",0.786089,0.306556,4.405968
925,2931,"(œuf, NOUN)",7,"[(7, 0.6858095715681806), (8, 0.11486580418186...",0.731771,0.314190,5.970529


In [47]:
dim_red = tod.dimension_reduction_classic.Tsne_corpus(corpus, n_components=2)
fig = tod.plotting.cluster_scatter_plot(corpus, dim_red, clustering)
fig

/opt/homebrew/lib/python3.11/site-packages/kaleido/__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [49]:
clustering.lexunit2cluster(('profond', 'ADJ'))

0

In [48]:
clustering.cluster2lexunit(0)

[('1er', 'ADJ'),
 ('2e', 'ADJ'),
 ('3e', 'ADJ'),
 ('XIIIe', 'ADJ'),
 ('XIIe', 'ADJ'),
 ('XIXe', 'ADJ'),
 ('XIe', 'ADJ'),
 ('XVIIIe', 'ADJ'),
 ('XVIIe', 'ADJ'),
 ('XVIe', 'ADJ'),
 ('XVe', 'ADJ'),
 ('XXe', 'ADJ'),
 ('ancien', 'ADJ'),
 ('autre', 'ADJ'),
 ('bas', 'ADJ'),
 ('beau', 'ADJ'),
 ('bon', 'ADJ'),
 ('bref', 'ADJ'),
 ('certain', 'ADJ'),
 ('cinquième', 'ADJ'),
 ('court', 'ADJ'),
 ('célèbre', 'ADJ'),
 ('dangereux', 'ADJ'),
 ('dernier', 'ADJ'),
 ('deuxième', 'ADJ'),
 ('différent', 'ADJ'),
 ('divers', 'ADJ'),
 ('double', 'ADJ'),
 ('dur', 'ADJ'),
 ('excellent', 'ADJ'),
 ('extrême', 'ADJ'),
 ('faible', 'ADJ'),
 ('fameux', 'ADJ'),
 ('faux', 'ADJ'),
 ('fort', 'ADJ'),
 ('futur', 'ADJ'),
 ('grand', 'ADJ'),
 ('grave', 'ADJ'),
 ('gros', 'ADJ'),
 ('haut', 'ADJ'),
 ('important', 'ADJ'),
 ('jeune', 'ADJ'),
 ('large', 'ADJ'),
 ('long', 'ADJ'),
 ('lourd', 'ADJ'),
 ('magnifique', 'ADJ'),
 ('mauvais', 'ADJ'),
 ('meilleur', 'ADJ'),
 ('moyen', 'ADJ'),
 ('multiple', 'ADJ'),
 ('même', 'ADJ'),
 ('net', 'AD

In [11]:
fig.write_html("sparse_kmeans_gsd.html")

In [39]:
def explain_sample(idx, clusters_above_threshold, top_n=10):
    x = Xd[idx]
    explanations = []
    for k in clusters_above_threshold:
        centroid = centroids[k]
        diff = x - centroid
        sq = diff ** 2
        total = sq.sum()

        if total == 0:
            explanations.append({
                "cluster": int(k),
                "membership": float(membership[idx, k]),
                "supportive_features": [],
                "opposing_features": []
            })
            continue

        contrib = sq / total if total > 0 else np.zeros_like(sq)
        informative = np.where(diff != 0)[0]
        if informative.size == 0:
            explanations.append({
                "cluster": int(k),
                "membership": float(membership[idx, k]),
                "supportive_features": [],
                "opposing_features": []
            })
            continue

        inf_contrib = contrib[informative]
        order = informative[np.argsort(inf_contrib)]
        # Order features by contribution (ascending)
        supportive_idx = order[:top_n]          # closest features
        opposing_idx = order[-top_n:][::-1]     # farthest features

        supportive = [(corpus.idx2feature(i), float(contrib[i])) for i in supportive_idx]
        opposing = [(corpus.idx2feature(i), float(contrib[i])) for i in opposing_idx]

        explanations.append({
            "cluster": int(k),
            "membership": float(membership[idx, k]),
            "supportive_features": supportive,
            "opposing_features": opposing
            })
    return explanations

all_explanations = []
for _, row in df_multi.iterrows():
    sample_row = int(row["row_idx"])
    lex_unit = row["lexunit"]
    clusters_above = [c for c, _m in row["clusters_above_threshold"]]
    detail = explain_sample(sample_row, clusters_above, top_n=10)
    all_explanations.append({
        "row_idx": sample_row,
        "lexunit": lex_unit,
        "clusters_above_threshold": clusters_above,
        "explanations": detail
    })

In [40]:
all_explanations

[{'row_idx': 0,
  'lexunit': ('$', 'NOUN'),
  'clusters_above_threshold': [2, 4, 5, 6, 7, 8],
  'explanations': [{'cluster': 2,
    'membership': 0.128206817996798,
    'supportive_features': [('node:X:next:VerbForm=Inf',
      2.3783646555743314e-10),
     ('node:X:parent:CxnElt=19:Conditional-NeutralEpistemic.Apodosis@p',
      2.3783646555743314e-10),
     ('node:X:own:CxnElt=7:Interrogative-Polar-Direct.Clause',
      2.3783646555743314e-10),
     ('node:X:own:Cxn=Interrogative-Polar-Direct', 2.3783646555743314e-10),
     ('node:X:own:rel_shallow=nsubj:caus', 2.3783646555743314e-10),
     ('node:X:parent:CxnElt=18:Conditional-NeutralEpistemic.Apodosis@p',
      2.3783646555743314e-10),
     ('node:X:child:rel_shallow=discourse', 2.3783646555743314e-10),
     ('node:X:child:upos=INTJ', 2.3783646555743314e-10),
     ('node:X:parent:CxnElt=7:Interrogative-WHInfo-Direct.Clause',
      2.3783646555743314e-10),
     ('node:X:next:Subject=ObjRaising', 2.3783646555743314e-10)],
    'opposi